# ZemaVoice Stage 9B — Kiki naturalness experiments
Compare gentle speech effects against the unmodified Amharic voice. **These are not genuine toddler voices.** Pitch and formant changes to a single adult TTS model cannot reliably produce a natural 3-year-old voice. The base `facebook/mms-tts-amh` model is CC BY-NC 4.0, for noncommercial evaluation only.

Run all cells in order. Choose the best sample by listening, not by its label.


In [ ]:
!pip -q install 'transformers>=4.44,<5' 'accelerate>=0.26' 'soundfile>=0.12' 'uroman>=1.3' 'librosa>=0.10' 'ipywidgets>=8'


In [ ]:
import numpy as np, torch, soundfile as sf, librosa
from IPython.display import display, Audio
from transformers import VitsModel, VitsTokenizer, set_seed
from uroman import Uroman
import ipywidgets as widgets
model_id='facebook/mms-tts-amh'
tokenizer=VitsTokenizer.from_pretrained(model_id)
model=VitsModel.from_pretrained(model_id)
romanizer=Uroman()
device='cuda' if torch.cuda.is_available() else 'cpu'
model=model.to(device).eval()
sr=int(model.config.sampling_rate)
print('Ready:',device,'sample rate:',sr)


In [ ]:
text_input=widgets.Textarea(value='ሰላም ጓደኞቼ! እንዴት ናችሁ? ዛሬ አብረን እንጫወታለን!',description='Amharic:',layout=widgets.Layout(width='95%',height='110px'))
display(text_input)


In [ ]:
raw=text_input.value.strip()
assert raw and len(raw)<=1000,'Enter 1 to 1000 Amharic characters.'
romanized=romanizer.romanize_string(raw)
assert romanized,'Romanization failed.'
inputs=tokenizer(romanized,return_tensors='pt').to(device)
set_seed(42)
with torch.no_grad():
    base=model(**inputs).waveform[0].detach().cpu().float().numpy().astype(np.float32)
print('Original length:',round(len(base)/sr,2),'seconds')
display(Audio(base,rate=sr))


In [ ]:
from pathlib import Path
out_dir=Path('/content/zemavoice_kiki_stage9b');out_dir.mkdir(exist_ok=True)
# Three gentle presets. Small shifts can sound more natural than extreme pitch shifts.
settings=[('original',0.0,1.00),('gentle',1.5,1.00),('bright',2.5,1.03),('soft',1.0,0.97)]
for name,steps,rate in settings:
    y=base.copy()
    if steps:
        y=librosa.effects.pitch_shift(y,sr=sr,n_steps=steps)
    if rate!=1.0:
        y=librosa.effects.time_stretch(y,rate=rate)
    y=np.clip(np.nan_to_num(y),-0.98,0.98).astype(np.float32)
    path=out_dir/f'kiki_{name}.wav'
    sf.write(path,y,sr)
    print(name,'|',round(len(y)/sr,2),'seconds |',path)
    display(Audio(y,rate=sr))


In [ ]:
import shutil
from google.colab import files
zip_path=shutil.make_archive('/content/zemavoice_kiki_stage9b_comparison','zip',out_dir)
files.download(zip_path)
